In [9]:
import scanpy as sc
import pandas as pd
import soupx
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import scrublet as scr
from io_utils import save_checkpoint

In [10]:
def estimate_ambient_rna(
        adata: sc.AnnData,
        adata_raw: sc.AnnData,
        cluster_key: str = "soupx_clusters",
        target_sum: float = 1e4,
        leiden_resolution: float = 0.4,
        n_pcs: int = 30,
        n_neighbors: int = 15
) -> tuple[sc.AnnData, object]:
    """
    Estimate ambient RNA contamination using SoupX.
    -----------
    Parameters
    -----------
        adata: High-quality cells AnnData object after removing low-quality cells.
        adata_raw: AnnData object imported from ".../Solo.out/GeneFull/raw", containing all droplets (cells + empty droplets).
        cluster_key: Name of the obs column in which helper SoupX clusters will be stored. 
        target_sum: Target library size for temporary normalisation, to be used in helper clustering.
        leiden_resolution: Resolution for helper Leiden clustering.
        n_pcs: Number of Principal components for helper clustering.
        n_neighbors: Number of neighbors for helper clustering.
    --------
    Returns
    --------
        adata: AnnData object, updated with helper cluster labels in 'adata.obs[cluster_key]'.
        sc_chan: SoupX Channel object with estimated contamination parameters.
    """

    # -----------------------------------------
    # 1. Prepare the raw count matrix
    # -----------------------------------------
    # Align genes between raw droplets and filtered cells.
    adata_raw = adata_raw[:, adata.var_names].copy()
    # Extra raw integer counts for the retained cells.
    adata_filtered_raw = adata_raw[adata.obs_names, :].copy()

    # -----------------------------------------
    # 2. Create the helper clustering for SoupX
    # -----------------------------------------
    adata_tmp = adata.copy()
    if "raw_counts" in adata_tmp.layers:
        adata_tmp.X = adata_tmp.layers["raw_counts"].copy()
    
    sc.pp.normalize_total(adata_tmp, target_sum = target_sum)
    sc.pp.log1p(adata_tmp)
    sc.pp.pca(adata_tmp, n_comps = max(n_pcs, 30))
    sc.pp.neighbors(adata_tmp, n_neighbors = n_neighbors, n_pcs = n_pcs)
    sc.tl.leiden(adata_tmp, resolution = leiden_resolution, key_added = cluster_key)

    adata.obs[cluster_key] = adata_tmp.obs[cluster_key].copy()
    del adata_tmp

    # -----------------------------------------
    # 3. Build SoupX Channel
    # -----------------------------------------
    sc_chan = soupx.SoupChannel(
        tod = adata_raw.X.T.tocsr(),
        toc = adata_filtered_raw.X.T.tocsr(),
        metaData = pd.DataFrame(index=adata.obs_names),
    )
    sc_chan.setClusters(adata.obs[cluster_key].astype(str).values)

    # -----------------------------------------
    # 4. Estimate ambient contamination
    # -----------------------------------------
    sc_chan = soupx.autoEstCont(sc_chan, verbose=True)

    print("Ambient RNA estimation complete.")

    return adata, sc_chan

In [3]:
def remove_ambient_rna(
    adata: sc.AnnData,
    sc_chan,
    layer_name: str = "soupx_counts",
    overwrite: bool = False
) -> sc.AnnData:
    """
    -----------
    Parameters
    -----------
        adata: AnnData object.
        sc_chan: SoupChannel object.
        layer_name: Name of the layer in which corrected counts will be stored. 
        overwrite: Whether to overwrite an existing layer with the same name.
    --------
    Returns
    --------
        adata: AnnData object updated with adata.layers[layer_name].
    """

    if layer_name in adata.layers and not overwrite:
        raise ValueError(
            f"adata.layers['{layer_name}'] already exists. "
            f"Set overwrite=True if you want to replace it. "
        )
    corrected_matrix = soupx.adjustCounts(sc_chan)
    adata.layers[layer_name] = corrected_matrix.T.copy()

    print(f"Ambient RNA correction applied. Correct counts stored in adata.layers['{layer_name}'].")

    original_sum = adata.layers["raw_counts"][0].sum() if "raw_counts" in adata.layers else adata.X[0].sum()
    corrected_sum = adata.layers[layer_name][0].sum()
    print(f"Orignal counts: {original_sum:.0f}")
    print(f"Corrected counts: {corrected_sum:.0f}")
    print(f"Counts removed: {original_sum - corrected_sum:.0f}")
    
    return adata

Below is the C9ALS sample going through the pipeline.

In [6]:
adata_dir = "../data/processed"
adata_als = sc.read_h5ad(f"{adata_dir}/c9ALS_GSM5292146_low_quality_filtered.h5ad")
adata_als_raw = sc.read_10x_mtx(
    "../results/40_starsolo_count/GSM5292146_c9ALS/Solo.out/GeneFull/raw",
    var_names="gene_symbols",
    cache=False)
adata_als_raw.var_names_make_unique()

adata_als, sc_chan = estimate_ambient_rna(adata_als, adata_als_raw)
adata_als = remove_ambient_rna(adata_als, sc_chan)


/opt/anaconda3/envs/c9_multiomics/lib/python3.11/site-packages/scanpy/readwrite.py:570: UserWarning: Suffix used (-[0-9]+) to deduplicate index values may make index values difficult to interpret. There values with a similar suffixes in the index. Consider using a different delimiter by passing `join={delimiter}`. Example key collisions generated by the make_index_unique algorithm: ['SNORD116-1', 'SNORD116-2', 'SNORD116-3', 'SNORD116-4', 'SNORD116-5']
  adata = _read_10x_mtx(


Collapsed to cluster level: (61552, 18) matrix
2752 genes passed tf-idf cut-off and 1210 soup quantile filter. Taking the top 100.
Found 836 usable gene×cluster combinations for estimation
Using 836 independent estimates of rho.
Estimated global rho of 0.05
Ambient RNA estimation complete.
Adjusting counts using method 'subtraction' with 18 clusters
Using subtraction method


/opt/anaconda3/envs/c9_multiomics/lib/python3.11/site-packages/scipy/sparse/_index.py:216: SparseEfficiencyWarning: Changing the sparsity structure of a csr_matrix is expensive. lil and dok are more efficient.
  self._set_arrayXarray(i, j, x)


Expanding counts from 18 clusters to 9168 cells (vectorized)
Ambient RNA correction applied. Correct counts stored in adata.layers['soupx_counts'].
Orignal counts: 3120
Corrected counts: 2964
Counts removed: 156


In [11]:
save_checkpoint(adata_als,
                "../data/processed", 
                "ambient_rna_removed") 

Checkpoint saved: ../data/processed/c9ALS_GSM5292146_ambient_rna_removed.h5ad


PosixPath('../data/processed/c9ALS_GSM5292146_ambient_rna_removed.h5ad')

In [ ]:
# Checks beore removing ambient RNA contamination.
# 1. any evidence of ambient RNA?


9168